<a href="https://colab.research.google.com/github/anbuchezhiyan2005/Image-Captioning-System/blob/main/notebooks/Image_Captioning_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU device: Tesla T4


In [2]:
!pip install -q torch torchvision pillow

In [3]:
import torch
import torchvision
from PIL import Image
import json
import os
import sys

print("All packages imported successfully!")

All packages imported successfully!


In [4]:
DATASET_PATH = "/content/drive/MyDrive/Image-Captioning-System/dataset"

print("Checking dataset files...")
print(f"Dataset path: {DATASET_PATH}")
print(f"Path exists: {os.path.exists(DATASET_PATH)}")

if os.path.exists(DATASET_PATH):
  print("\n Dataset files in the folder: ")
  for item in os.listdir(DATASET_PATH):
    print(item)

  captions_file = os.path.join(DATASET_PATH, "captions_encoded.json")
  vocab_file = os.path.join(DATASET_PATH, "vocabulary.json")
  images_directory = os.path.join(DATASET_PATH, "Processed_Images")

  print(f"\n✅ captions_encoded.json exists: {os.path.exists(captions_file)}")
  print(f"✅ vocabulary.json exists: {os.path.exists(vocab_file)}")
  print(f"✅ Processed_Images/ exists: {os.path.exists(images_directory)}")

  if os.path.exists(images_directory):
      num_images = len(os.listdir(images_directory))
      print(f"✅ Number of images: {num_images}")
  else:
      print("❌ Dataset path not found!")
      print("Please upload your dataset to Google Drive first.")


Checking dataset files...
Dataset path: /content/drive/MyDrive/Image-Captioning-System/dataset
Path exists: True

 Dataset files in the folder: 
vocabulary.json
captions_encoded.json
Processed_Images

✅ captions_encoded.json exists: True
✅ vocabulary.json exists: True
✅ Processed_Images/ exists: True
✅ Number of images: 8091


In [5]:
PROJECT_ROOT = "/content/Image-Captioning-System"
sys.path.insert(0, PROJECT_ROOT)

DATASET_PATH = "/content/drive/MyDrive/Image-Captioning-System/dataset"

CAPTIONS_FILE = os.path.join(DATASET_PATH, "captions_encoded.json")
VOCAB_FILE = os.path.join(DATASET_PATH, "vocabulary.json")
IMAGE_DIR = os.path.join(DATASET_PATH, "Processed_Images")

print(f"\n✅ captions_encoded.json exists: {os.path.exists(CAPTIONS_FILE)}")
print(f"✅ vocabulary.json exists: {os.path.exists(VOCAB_FILE)}")
print(f"✅ Processed_Images/ exists: {os.path.exists(IMAGE_DIR)}")


✅ captions_encoded.json exists: True
✅ vocabulary.json exists: True
✅ Processed_Images/ exists: True


In [6]:
from models.encoder import CNNEncoder
from models.decoder import LSTMDecoder
from training.dataset import MyDataset, collate_fn
from training.train_config import get_loss_function, get_optimizer

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm

with open(VOCAB_FILE, mode = "r") as f:
  vocab_data = json.load(f)
  VOCAB_SIZE = len(vocab_data)
  print(f"✅ Vocabulary size: {VOCAB_SIZE}")

✅ Vocabulary size: 8832


In [7]:
print("Creating dataset...")
dataset = MyDataset(CAPTIONS_FILE, IMAGE_DIR)
print(f"Created Dataset: {len(dataset)} samples")

# Creating DaaLoader
BATCH_SIZE = 32
NUM_WORKERS = 2

dataloader = DataLoader(
    dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = collate_fn,
    num_workers = NUM_WORKERS,
    pin_memory =True
)

print(f"✅ DataLoader created: {len(dataloader)} batches")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Total samples: {len(dataset)}")

# Testing one batch
images, captions = next(iter(dataloader))
print(f"✅ Batch images shape: {images.shape}")
print(f"✅ Batch captions shape: {captions.shape}")

Creating dataset...
Created Dataset: 40455 samples
✅ DataLoader created: 1265 batches
   Batch size: 32
   Total samples: 40455
✅ Batch images shape: torch.Size([32, 3, 224, 224])
✅ Batch captions shape: torch.Size([32, 20])


In [8]:
# Defining Hyperparameters
EMBED_SIZE = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 1
LEARNING_RATE = 0.001

# Device Configuration
device = torch.device('cuda'if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

# Initialize models
print("Initializing models...")
encoder = CNNEncoder().to(device)
decoder = LSTMDecoder(
    embedding_dim = EMBED_SIZE,
    hidden_size = HIDDEN_SIZE,
    vocab_size = VOCAB_SIZE,
    num_layers = NUM_LAYERS
).to(device)

print(f"✅ Encoder initialized (embed_size={EMBED_SIZE})")
print(f"✅ Decoder initialized (hidden={HIDDEN_SIZE}, layers={NUM_LAYERS}, vocab={VOCAB_SIZE})")

# Count parameters
encoder_params = sum(p.numel() for p in encoder.parameters())
decoder_params = sum(p.numel() for p in decoder.parameters())
total_params = encoder_params + decoder_params

print(f"\n📊 Model Parameters:")
print(f"   Encoder: {encoder_params:,}")
print(f"   Decoder: {decoder_params:,}")
print(f"   Total: {total_params:,}")


✅ Device: cuda
Initializing models...
✅ Encoder initialized (embed_size=256)
✅ Decoder initialized (hidden=512, layers=1, vocab=8832)

📊 Model Parameters:
   Encoder: 23,508,032
   Decoder: 9,417,856
   Total: 32,925,888


In [11]:
# Loss Function
criterion = get_loss_function()
print(f"✅ Loss function: {criterion}")

params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = get_optimizer(params, LEARNING_RATE)

print(f"✅ Loss function: CrossEntropyLoss (ignore_index=0)")
print(f"✅ Optimizer: Adam (lr={LEARNING_RATE})")
print(f"✅ Total trainable parameters: {sum(p.numel() for p in params if p.requires_grad):,}")

✅ Loss function: CrossEntropyLoss()
✅ Loss function: CrossEntropyLoss (ignore_index=0)
✅ Optimizer: Adam (lr=0.001)
✅ Total trainable parameters: 32,925,888


In [ ]:
def train_epoch(
    encoder,
    decoder,
    dataloader,
    criterion,
    optimizer,
    device
):

  # Setting models to training mode
  encoder.train()
  decoder.train()

  total_loss = 0
  num_batches = len(dataloader)

  # Implementing progress bar(tqdm)
  pbar = tqdm(dataloader, desc = "Training", unit = "batches")

  for (images, captions) in pbar:

    # moving images and captions to GPU
    images = images.to(device)
    captions = captions.to(device)

    # Setting gradients to zero
    optimizer.zero_grad()

    # Forward pass - getting features from the encoder
    features = encoder(images)

    # Forward pass - getting the outputs from the decoder
    outputs = decoder(features, captions)

    # reshaping outputs and captions for loss calculation
    outputs = outputs.view(-1, outputs.size(-1))
    captions = captions.view(-1)

    # Calculate the loss
    loss = criterion(outputs, captions)

    # Backward pass
    loss.backward()

    # gradient clipping (prevents exploding gradients)
    torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm = 5.0)
    torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm = 5.0)

    # Updating weights
    optimizer.step()

    # Updating total loss
    total_loss += loss.item()

    # tracking the loss
    pbar.set_postfix({'loss': f'{loss.item():.4f}'})

  avg_loss = total_loss / num_batches
  return avg_loss

print("Training loop defined!")




